# Get PAM sites in whole-genome non-coding regions

This notebook uses python package `pybedtools` to parse CREs, and `re` to find positions of NGG PAMs.

This serves as the first layer of reference genome predictions


In [1]:
# move up one-level to access backend
%cd ..

/common/wut4/workspace/CROTONdb


In [2]:
import os
import json
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

from pyfaidx import Fasta
import pybedtools

## Step 1: retrieve data

In [4]:
CRE_DB_PATH = './frontend/data/genomes/GRCh38-cCREs.PLS.bed'

In [5]:
bed_file = pybedtools.BedTool(CRE_DB_PATH)

cre_df = bed_file.to_dataframe()

# Manually assign column name for CRCH38-cCREs
cre_df.columns = ['chrom', 'start', 'end', 'accession', 'genename', 'annotation']
# Please be notice that we use "genename" for "cCRE IDs" so that the code can find it
cre_df = cre_df[cre_df['chrom'] != 'chrY']

In [6]:
cre_df.head()

,chrom,start,end,accession,genename,annotation
0,chr1,778570,778919,EH38D4327580,EH38E2776539,"PLS,CTCF-bound"
1,chr1,779026,779180,EH38D4327581,EH38E2776540,"PLS,CTCF-bound"
2,chr1,817080,817403,EH38D2115333,EH38E1310166,PLS
3,chr1,827417,827767,EH38D4327609,EH38E2776555,"PLS,CTCF-bound"
4,chr1,904578,904918,EH38D4327691,EH38E2776597,"PLS,CTCF-bound"


In [7]:
# Add strand on both direction
df_plus = cre_df.copy()
df_plus['strand'] = '+'

df_minus = cre_df.copy()
df_minus['strand'] = '-'

cre_df = pd.concat([df_plus, df_minus]).reset_index(drop=True)
print(len(cre_df))

81696


## Step 2: add FA information to get sequence

In [8]:
assert os.path.isfile(backend.configs.GENOME_FA_PATH), f"Couldn't find FA file: {backend.configs.GENOME_FA_PATH}"

In [9]:
cre_df = backend.get_CDS_seq(df=cre_df, genome_path=backend.configs.GENOME_FA_PATH)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 81696/81696 [00:03<00:00, 21896.29it/s]


In [10]:
cre_df.head()

,chrom,start,end,accession,genename,annotation,strand,seq
0,chr1,778570,778919,EH38D4327580,EH38E2776539,"PLS,CTCF-bound",+,TGATGAGAAAACTGCCCAGCTCCAGGCACCATGGCGCCCCAGTGAT...
1,chr1,779026,779180,EH38D4327581,EH38E2776540,"PLS,CTCF-bound",+,CAGAAGCTCCTCAATGGCCAGCGCCAGCTGCAGCCCCGGCCGCCCA...
2,chr1,817080,817403,EH38D2115333,EH38E1310166,PLS,+,CTTGGCAAAGAATTTTTGGCTAAGTTCCCAAAAACGATTGCAACAA...
3,chr1,827417,827767,EH38D4327609,EH38E2776555,"PLS,CTCF-bound",+,GTGGAGGACGCCGCAGGGAGGGGACTGCGTGGCTGGGTTTGGCCAC...
4,chr1,904578,904918,EH38D4327691,EH38E2776597,"PLS,CTCF-bound",+,CTCCTCCGAACGTGGCCGCCTCCTCCTCCGAACGTGGCCTCCTCCG...


## Step 3: Find PAM location by searching for NGG

In [11]:
pam_df = backend.get_PAM_coords(df=cre_df)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 81696/81696 [00:07<00:00, 10581.25it/s]


In [12]:
pam_df.head()

,chrom,start,end,accession,genename,annotation,strand,seq,pams,rc_pams
0,chr1,778570,778919,EH38D4327580,EH38E2776539,"PLS,CTCF-bound",+,TGATGAGAAAACTGCCCAGCTCCAGGCACCATGGCGCCCCAGTGAT...,"[778593, 778699, 778721, 778781, 778862, 77889...","[778808, 778710, 778685, 778659, 778636, 77888..."
1,chr1,779026,779180,EH38D4327581,EH38E2776540,"PLS,CTCF-bound",+,CAGAAGCTCCTCAATGGCCAGCGCCAGCTGCAGCCCCGGCCGCCCA...,"[779123, 779062, 779129, 779139, 779151, 77909...","[779175, 779160, 779114, 779090, 779079, 77903..."
2,chr1,817080,817403,EH38D2115333,EH38E1310166,PLS,+,CTTGGCAAAGAATTTTTGGCTAAGTTCCCAAAAACGATTGCAACAA...,"[817251, 817273, 817334, 817340, 817397, 81727...","[817390, 817378, 817375, 817257, 817210, 81738..."
3,chr1,827417,827767,EH38D4327609,EH38E2776555,"PLS,CTCF-bound",+,GTGGAGGACGCCGCAGGGAGGGGACTGCGTGGCTGGGTTTGGCCAC...,"[827421, 827431, 827435, 827472, 827540, 82756...","[827672, 827642, 827495, 827748, 827740, 82768..."
4,chr1,904578,904918,EH38D4327691,EH38E2776597,"PLS,CTCF-bound",+,CTCCTCCGAACGTGGCCGCCTCCTCCTCCGAACGTGGCCTCCTCCG...,"[904757, 904775, 904902, 904628, 904650, 90467...","[904871, 904807, 904794, 904725, 904722, 90470..."


## Step 4: transfer found pams and rc_pams location to rows, then add pam id

In [13]:
# to make 60bp input for CROTON, we pad 33bp to the left of PAM, and 27bp to the right
CREpamsbed_df, skipped_genes = backend.make_CDSpamsbed(pam_df=pam_df, pam_left=33, pam_right=27)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40848/40848 [02:03<00:00, 330.55it/s]


processed PAMs in CDS: (2436653, 7)
skipped genes: []


In [14]:
CREpamsbed_df.head()

,start,end,strand,#,genename,num,pamid
21,778560,778620,-,1,EH38E2776539,1,EH38E2776539|1
56,778560,778620,+,1,EH38E2776539,2,EH38E2776539|2
22,778561,778621,-,1,EH38E2776539,3,EH38E2776539|3
23,778567,778627,-,1,EH38E2776539,4,EH38E2776539|4
60,778568,778628,+,1,EH38E2776539,5,EH38E2776539|5


## Step 5: Re-getting sequence from FA file

In [15]:
CREpamsbed_df = backend.get_ref_PAM_seq(df=CREpamsbed_df, genome_path=backend.configs.GENOME_FA_PATH)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 2436653/2436653 [01:37<00:00, 24942.43it/s]


In [16]:
CREpamsbed_df.head()

,start,end,strand,#,genename,num,pamid,ref_seq
21,778560,778620,-,1,EH38E2776539,1,EH38E2776539|1,CTACATCACTGGGGCGCCATGGTGCCTGGAGCTGGGCAGTTTTCTC...
56,778560,778620,+,1,EH38E2776539,2,EH38E2776539|2,GTCCCCACTCTGATGAGAAAACTGCCCAGCTCCAGGCACCATGGCG...
22,778561,778621,-,1,EH38E2776539,3,EH38E2776539|3,GCTACATCACTGGGGCGCCATGGTGCCTGGAGCTGGGCAGTTTTCT...
23,778567,778627,-,1,EH38E2776539,4,EH38E2776539|4,TGTTCGGCTACATCACTGGGGCGCCATGGTGCCTGGAGCTGGGCAG...
60,778568,778628,+,1,EH38E2776539,5,EH38E2776539|5,TCTGATGAGAAAACTGCCCAGCTCCAGGCACCATGGCGCCCCAGTG...


## Step 6: Split the data by chrom, then save the file
If we only process one chrom, will get empty on other file

In [17]:
CRE_PAM_DIR = './frontend/data/bed/CRE_PLS_pams-byChrom/'
if not os.path.isdir(CRE_PAM_DIR):
    os.makedirs(CRE_PAM_DIR)

In [18]:
backend.subset_CDSpams(CREpamsbed_df, save_path=CRE_PAM_DIR, chrom_type='number')

X
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
